In [0]:
# Data Quality Checks after Gold Layer Build
from pyspark.sql.functions import col, lit, countDistinct

# 1. Confirm all 8 source tables ingested into Bronze
bronze_tables = [
    "airline", "booking", "flight_booking", "nation", "passenger", "region", "route", "airline_route"
]
catalog = "tpc_a_dev"  # adjust if needed
bronze_schema = f"{catalog}.bronze"
missing_bronze = []
for tbl in bronze_tables:
    if not spark.catalog.tableExists(f"{bronze_schema}.{tbl}"):
        missing_bronze.append(tbl)
if missing_bronze:
    raise Exception(f"Missing Bronze tables: {missing_bronze}")
print("All bronze tables exist.")

# 2. PK populated & unique in each Silver table
silver_tables_keys = {
    "airline": "al_airlineid",
    "booking": "bk_bookingid",

    "nation": "n_nationkey",
    "passenger": "pa_passengerid",
    "region": "r_regionkey",
    "route": "ro_routeid",
    "airline_route": ["ar_airlineid", "ar_routeid"]
}
silver_schema = f"{catalog}.silver"
for tbl, key in silver_tables_keys.items():
    df = spark.table(f"{silver_schema}.{tbl}")
    if isinstance(key, list):
        nulls = df.filter(col(key[0]).isNull() | col(key[1]).isNull()).count()
        dups = df.groupBy(key).count().filter(col("count") > 1).count()
        if nulls or dups:
            raise Exception(f"PK check failed for {tbl}: nulls={nulls}, dups={dups}")
    else:
        nulls = df.filter(col(key).isNull()).count()
        dups = df.groupBy(key).count().filter(col("count") > 1).count()
        if nulls or dups:
            raise Exception(f"PK check failed for {tbl}: nulls={nulls}, dups={dups}")
print("Silver PK checks passed.")

# 3. Fact table contains no null surrogate keys
fact = spark.table(f"{catalog}.gold.fact_bookings")
surrogate_keys = ["route_id", "airline_id", "booking_id", "booking_line"]
for key in surrogate_keys:
    nulls = fact.filter(col(key).isNull()).count()
    if nulls:
        raise Exception(f"Fact_bookings null surrogate key: {key}")
print("Fact surrogate key null checks passed.")

# 4. Numeric measures business rules
n_invalid = fact.filter(
    (col("ticket_price") <= 0) | (col("discount_pct") < 0) | (col("discount_pct") > 1) | (col("seats") < 1)
).count()
if n_invalid:
    raise Exception(f"Fact_bookings business rule violation (invalid measures): {n_invalid}")
print("Fact numeric measure checks passed.")

# 5. Gold/Silver reconciliation: revenue and row counts
rec = spark.table(f"{catalog}.gold.reconciliation_summary").collect()[0]
rev_g = rec["gold_revenue"]
rev_s = rec["silver_revenue"]
rows_g = rec["gold_row_count"]
rows_s = rec["silver_row_count"]
rev_diff = abs(rev_g - rev_s)/max(rev_g, rev_s)
row_diff = abs(rows_g - rows_s)/max(rows_g, rows_s)
if rev_diff > 0.0001:
    raise Exception(f"Revenue mismatch: gold={rev_g}, silver={rev_s}, tolerance exceeded ({rev_diff*100:.4f}%)")
if row_diff > 0.0001:
    raise Exception(f"Row count mismatch: gold={rows_g}, silver={rows_s}, tolerance exceeded ({row_diff*100:.4f}%)")
print("Gold/Silver reconciliation checks passed.")

# 6. Rejected record counts per table
quarantine_suffix = "_quarantine"
rejected_report = {}
for tbl in bronze_tables:
    quarantine_tbl = f"{silver_schema}.{tbl}{quarantine_suffix}" if tbl not in ["region", "route"] else f"{silver_schema}.{tbl}_quarantine"
    if spark.catalog.tableExists(quarantine_tbl):
        rejected_report[tbl] = spark.table(quarantine_tbl).count()
    else:
        rejected_report[tbl] = 0
print("Rejected records by table:", rejected_report)
print("ALL CHECKS PASSED.")